# 03 — Summaries Review

Прогон суммаризации для четырёх характерных точек и просмотр результатов.

| Тип | run_date |
|-----|----------|
| calm_2021 | 2021-10-20 |
| shock_war | 2022-03-25 |
| shock_covid | 2020-04-20 |
| calm_2023 | 2023-05-01 |

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

print("CWD:", os.getcwd())
print("OPENROUTER_API_KEY:", "OK" if os.environ.get("OPENROUTER_API_KEY") else "НЕ НАЙДЕН")

CWD: /home/ustyantsev/workspace/blind_prophet
OPENROUTER_API_KEY: OK


In [2]:
from amnesiac.store.db import get_connection
from amnesiac.summarize.runner import run_summarize_pipeline

DB_PATH = Path("data/db/blind_prophet.db")

PERIODS = {
    "calm_2021":   "2021-10-20",
    "shock_war":   "2022-03-25",
    "shock_covid": "2020-04-20",
    "calm_2023":   "2023-05-01",
}
label_map = {v: k for k, v in PERIODS.items()}

## S01 — Прогон

Установи `FORCE = True` чтобы перезаписать уже существующие периоды.

In [3]:
FORCE = True  # True — перезаписать все; False — пропустить существующие

conn = get_connection(DB_PATH)
existing = {row[0] for row in conn.execute("SELECT run_date FROM summaries")}
conn.close()

if FORCE:
    to_run = PERIODS
else:
    to_run = {name: date for name, date in PERIODS.items() if date not in existing}

print("Уже в DB: ", sorted(existing) or "—")
print("К прогону:", list(to_run.keys()) or "—")
print("FORCE:", FORCE)

Уже в DB:  ['2020-04-20', '2021-10-20', '2023-05-01']
К прогону: ['calm_2021', 'shock_war', 'shock_covid', 'calm_2023']
FORCE: True


In [4]:
for name, date in to_run.items():
    print(f"▶ {name} ({date})...")
    await run_summarize_pipeline(DB_PATH, date, force=FORCE)
    print(f"✓ {name} готово\n")

▶ calm_2021 (2021-10-20)...
продовольствие: 28.7s
тарифы: 33.2s
бюджет: 35.9s
труд: 38.2s
дкп: 45.4s
инфляция: 46.6s
кризис: 52.9s
курс: 53.6s
зарплаты: 68.8s
	meta: 47.1s
✓ calm_2021 готово

▶ shock_war (2022-03-25)...
продовольствие: 34.7s
зарплаты: 35.2s
тарифы: 36.5s
труд: 43.1s
курс: 49.2s
дкп: 71.6s
кризис: 93.3s
инфляция: 94.2s
бюджет: 292.6s
	meta: 88.8s
✓ shock_war готово

▶ shock_covid (2020-04-20)...
продовольствие: 29.9s
кризис: 44.4s
труд: 52.6s
дкп: 60.7s
курс: 66.9s
бюджет: 75.3s
зарплаты: 78.7s
тарифы: 79.2s
инфляция: 82.0s
	meta: 82.2s
✓ shock_covid готово

▶ calm_2023 (2023-05-01)...
курс: 37.0s
дкп: 55.5s
тарифы: 72.4s
продовольствие: 73.3s
труд: 79.9s
зарплаты: 124.5s
бюджет: 153.1s
инфляция: 218.2s
кризис: 485.9s
	meta: 79.2s
✓ calm_2023 готово



## S02 — Статистика

In [5]:
conn = get_connection(DB_PATH)
rows = conn.execute(
    "SELECT run_date, doc_count, model, length(summary) FROM summaries ORDER BY run_date"
).fetchall()
conn.close()

print(f"{'run_date':<14} {'label':<14} {'docs':>5} {'chars':>6}  model")
print("-" * 62)
for run_date, doc_count, model, chars in rows:
    print(f"{run_date:<14} {label_map.get(run_date, '?'):<14} {doc_count:>5} {chars:>6}  {model}")

run_date       label           docs  chars  model
--------------------------------------------------------------
2020-04-20     shock_covid      429  13900  deepseek/deepseek-v4-flash
2021-10-20     calm_2021        411   8860  deepseek/deepseek-v4-flash
2022-03-25     shock_war        365  13361  deepseek/deepseek-v4-flash
2023-05-01     calm_2023        323  13252  deepseek/deepseek-v4-flash


## S03 — Саммари

In [6]:
def show(run_date: str):
    conn = get_connection(DB_PATH)
    row = conn.execute("SELECT summary FROM summaries WHERE run_date = ?", (run_date,)).fetchone()
    conn.close()
    label = label_map.get(run_date, run_date)
    display(Markdown(f"---\n## {label} — {run_date}\n\n{row[0] if row else 'нет данных'}"))

In [7]:
show("2020-04-20")

---
## shock_covid — 2020-04-20

# Единый нарративный таймлайн: 14-дневный период экономического кризиса

## Узел 1. Ослабление рубля, нефтяной шок и валютная волатильность (дни 1–5, 10)

Период открывается официальными курсами ЦБ на день 1: доллар — 76,41 руб., евро — 82,63 руб. Уже на день 2 Минэкономразвития фиксирует ослабление рубля к корзине основных валют за предыдущий месяц на 15,9%. Фон валютного рынка задаёт крайне нестабильная нефтяная конъюнктура. На день 2 фьючерс Brent кратковременно превышает $34/барр., после чего снижается; Минэнерго США понижает прогноз средней цены Brent на текущий год до $33/барр., на следующий — до $45,6/барр. Цена российской нефти Urals на начало месяца достигает минимума с 1999 года — $10,54/барр.

В середине первой недели (день 4) на фоне новостей о переговорах по нефти рубль укрепляется: курс доллара опускается ниже 74 руб., евро — ниже 80 руб. (официально: 74,61 и 81,09 соответственно). Однако высокая волатильность сохраняется: Brent колеблется от +10% до –3% за день. Лидеры России, США и Саудовской Аравии подтверждают настрой на координацию. На день 5 цена Urals превышает $20/барр., но это временное улучшение.

Выход из периода (день 10) приносит резкое ухудшение. После заключения нового соглашения ОПЕК+ (день 8, о котором ниже) средняя цена Urals за предыдущий месяц снижается до $19/барр. против $47,3 месяцем ранее, а Urals дешевеет на $3,59 до $16,71/барр. Пошлина на экспорт нефти из РФ со следующего месяца снижается в 7,6 раза — до $6,8 за тонну. Курс доллара на Московской бирже поднимается до 75 руб., евро — до 80,57 руб. Реальный эффективный курс рубля за прошедшие три месяца снизился на 11,5%, в том числе за последний месяц — на 11,4%. Индекс Мосбиржи падает более чем на 5% (ниже 2500 пунктов). Нефть Brent падает на 5% после прогнозов МЭА по спросу (опускается ниже $28/барр.). На день 12 нефть WTI обновляет многолетний минимум, опускаясь ниже $19/барр. (по другим данным — ниже $18/барр., минимум с 2002 года).

**Затронутые темы:** валютный курс, нефтяной рынок, санкционное давление, сделка ОПЕК+, внешняя торговля, макроэкономические прогнозы.
**Причинно-следственные связи:** резкое падение нефтяных цен → ослабление рубля → ускорение инфляции (следующий узел). Сделка ОПЕК+ (день 8) не останавливает падение Urals из-за коллапса спроса.

## Узел 2. Инфляционный всплеск и продовольственные цены (дни 1–9)

Первая половина периода характеризуется резким ускорением роста цен на базовые продукты. На день 1 Росстат публикует данные за предыдущий месяц: сахар подорожал на 13,5%, гречка — на 6,2%, макароны — на 2,4%, чеснок — на 8,9%, лимоны — на 7,8%, ртутные градусники — на 3,2%. ФАС получает жалобы на завышение цен на имбирь, чеснок, лимоны и мёд, связывая это с ограничением поставок и ростом спроса на фоне пандемии.

На день 3 Росстат сообщает о продолжающемся активном росте цен за неделю: гречка +3,5%, яйца +3,2%, сахар +3,1%. Одновременно правительство на день 1 ввело государственное регулирование цен на медицинские маски и перчатки: розничная наценка не более 10 коп. за штуку, на всех этапах производства — не более 10%.

На день 4 вице-премьер Абрамченко заявляет, что сахар сейчас дешевле, чем год назад (35 руб./кг против 46 руб./кг), — однако это сравнение не отменяет резкого месячного роста. На день 5 ФАС предлагает заморозить индексацию оптовых цен на газ из-за коронавируса (ранее на день 2 предлагалось индексировать на 3%). На день 7 ФАС констатирует стабилизацию розничных цен на имбирь после скачка, вызванного слухами о профилактике COVID-19.

Вторая неделя (дни 8–9) приносит новые регуляторные инициативы. Генпрокурор поручает реагировать на взвинчивание цен на муку, гречку и аналогичные товары в регионах. Депутаты Госдумы предлагают ограничить наценку на социально значимые продукты (мясо, рыба, молоко, яйца, хлеб, крупы, картофель); участники рынка предупреждают, что эта мера грозит дефицитом и ростом нелегального бизнеса. Фиксируется недельная динамика цен: лук +13,1%, картофель +3,5%, огурцы –8,7%, помидоры –5,3%. К концу периода отмечен резкий рост цен на кофейные зёрна из-за начала активного запасания импортёрами и потребителями.

На день 10 президент Путин сообщает о сокращении розничной торговли на 35% из-за пандемии. Средний чек покупок, по данным «Ромир», на первой нерабочей неделе упал на 20% до 582 руб.; по данным ВТБ — на 17%. Рост спроса фиксируется на отдельные категории: алкоголь (+26%), сигареты (+15%), газировка (+47%), пицца (+71%).

**Затронутые темы:** инфляция, продовольственная безопасность, госрегулирование цен, потребительский спрос, санитарные товары.
**Причинно-следственные связи:** ослабление рубля (узел 1) → рост импортных товаров и сырья → ускорение продовольственной инфляции. Рост спроса на отдельные товары из-за панических настроений и введения самоизоляции.

## Узел 3. Сигналы денежно-кредитной политики и банковская система (дни 5–12)

На день 5 ЦБ подаёт множественные сигналы: снижение ключевой ставки будет одним из вариантов на следующем заседании совета директоров; ситуация на российском финансовом рынке начала улучшаться, необходимости в валютных интервенциях нет; темпы выдачи беспроцентных зарплатных кредитов не соответствуют остроте проблемы; ЦБ не видит массового повышения ставок по ипотеке, но ожидает некоторого ухудшения качества обслуживания кредитов; динамика ВВП в текущем году будет отрицательной.

В тот же день Минфин приобретает у ЦБ 50% акций Сбербанка за 2,139 трлн руб. по цене 189,44 руб. за акцию. Доходы от сделки направляются на бюджетные цели. Это крупнейшая внутрироссийская сделка по передаче актива.

На день 8 системно значимый банк (Сбербанк) открывает в ЦБ безотзывную кредитную линию на 500 млрд руб. для соблюдения норматива краткосрочной ликвидности.

На день 10 максимальная ставка по вкладам топ-10 банков составляет 5,43% (рост в течение двух декад подряд). Анонсируется новый спецкредитный продукт для системообразующих предприятий: ставка будет субсидироваться на ключевую ставку ЦБ, половина кредита — под гарантии Минфина. Внешэкономбанк гарантирует 75% зарплатных кредитов.

На день 11 анонсируется программа ипотеки с субсидированной ставкой 6,5% (сокращает ежемесячный платёж примерно на 20%). Заявки можно подавать до установленного срока. Москва выделяет 20–30 млрд руб. на субсидирование ставок для малого бизнеса. Выдача кредитных карт за первый квартал упала на 18% в годовом выражении, лимиты — на 28,6%. ЦБ рекомендовал банкам до определённого срока обслуживать клиентов с просроченными паспортами.

На день 12 глава ЦБ заявляет, что на следующей неделе будет предметно рассмотрен вопрос о снижении ключевой ставки; повышение маловероятно. ЦБ утверждает новые меры по поддержке экономики, обещает к середине периода принять меры для снижения ставок по ипотеке. Рефинансировать уже имеющуюся ипотеку по ставке 6,5% невозможно. ЦБ заявляет, что доля ипотеки в экономике пока невелика, но анонсированные меры поддержат рынок. Прямые выплаты населению («вертолётные деньги») как инструмент ДКП неактуальны.

**Затронутые темы:** денежно-кредитная политика, ставки, ипотека, ликвидность банковской системы, сделка по Сбербанку, кредитование, госгарантии.
**Причинно-следственные связи:** острый кризис ликвидности (узел 4) → необходимость увеличения ликвидности через сделку со Сбербанком и кредитную линию. Падение доходов населения → ухудшение качества кредитов → анонс субсидирования ставок.

## Узел 4. Рынок труда, доходы и меры поддержки занятости (дни 1–12)

Рынок труда входит в период с резкими сигналами. На день 1 власти Москвы сообщают о возобновлении работы трети промышленных предприятий города, где заняты 160 тыс. человек. Роструд разъясняет: работающие в нерабочие дни получают обычную зарплату, не повышенную. Международная организация труда прогнозирует потери рынка труда, которые могут превысить 25 млн рабочих мест.

На день 3 сервисы по поиску работы фиксируют, что зарплаты курьеров практически не изменились; максимальная «вилка» достигает 110–150 тыс. руб. в месяц. Минтруд анонсирует дополнительные пособия по 5 тыс. руб. на 3,3 млн российских детей. Правительство объявляет о выплате потерявшим работу трёх максимальных пособий по безработице (более 12 тыс. руб. ежемесячно) в течение апреля–июня.

На день 5 замминистра Москвы заявляет о сокращении численности работников на предприятиях города с 3 млн до 300 тыс. — десятикратное падение. Число вакансий для удаленной работы выросло на 71% относительно предыдущего месяца; максимальные зарплаты на «удаленке» достигают 80 тыс. руб. (для психологов) и выше (продажи, телемаркетинг). Розничная торговля сокращается на 35%.

На день 8 глава Счётной палаты Кудрин прогнозирует временный рост числа безработных в России с 2,5 до 8 млн и призывает увеличить пособия по безработице и МРОТ. Правительство выделяет 50 млрд руб. на дополнительные выплаты медикам. Кудрин оценивает рост безработицы втрое и падение ВВП на 5%.

На день 10 президент объявляет о новой поддержке: деньги на зарплаты в размере около 12 тыс. руб. на сотрудника. Минтруд сообщает о росте числа безработных до 735 тыс. (+44 тыс. с начала года). «Ведомости» отмечают, что бизнес ожидал большей поддержки (один МРОТ — это около 25% от средней зарплаты). Президент поручает обеспечить врачам обещанные выплаты до 80 тыс. руб. чистыми. Минтруд: 7,5% работников находятся на удалёнке; более 1 млн семей обратились за ежемесячной выплатой 5 тыс. руб. из материнского капитала.

На день 11 премьер-министр сообщает, что прямые выплаты государства затронут полмиллиона компаний (около 3 млн человек). Минэкономразвития уточняет: выплаты малому и среднему бизнесу покроют чуть больше половины зарплат сотрудников в апреле–мае. Национальное рейтинговое агентство оценивает ущерб экономике РФ в 18 трлн руб., а число потенциально оставшихся без работы — 15,5 млн человек. «Ведомости»: на доплату к зарплате смогут рассчитывать более 3 млн человек, но это только половина занятых в МСБ; многие не смогут претендовать из-за требования сохранения 90% сотрудников.

На день 12 Мишустин подписывает постановление о максимальных выплатах безработным — 12 130 руб. в месяц. Один из крупнейших работодателей (РЖД) приостанавливает сокращение персонала. Опросы фиксируют падение доходов у 49% россиян, 46% опасаются нехватки денег на еду, 42% урезают расходы; 74% считают, что найти работу будет сложно.

**Затронутые темы:** безработица, доходы населения, пособия, зарплатные кредиты, медицина, удалённая работа, секторальный кризис (фитнес, туризм, строительство).
**Причинно-следственные связи:** падение доходов → сокращение потребительского спроса (узел 2). Ограничительные меры → резкое падение занятости в сфере услуг. Бюджетные выплаты лишь частично компенсируют потери.

## Узел 5. Бюджетные антикризисные меры и фискальная политика (дни 1–12)

Период насыщен анонсами и реализацией масштабных бюджетных расходов. На день 1 Минтруд объявляет о выделении почти 200 млрд руб. на поддержку семей с детьми (в том числе 140 млрд — на детей от 3 до 7 лет), 30 млрд руб. на пособия по безработице и около 35 млрд руб. на изменённый порядок оплаты больничных. Правительство утверждает два сценария стресс-тестов для системообразующих компаний: изоляция при нефти $20/барр. до конца июня или жёсткий карантин при $10/барр. до конца сентября.

На день 3 премьер-министр заявляет, что из-за пандемии страна недополучит нефтегазовые и ненефтегазовые доходы — серьёзный удар по бюджету. На день 5 президент поручает Минфину обеспечить регионам гибкость в распределении антикризисных средств. Регионам выделяется 33 млрд руб. на оснащение больниц.

На день 8 правительство выделяет на борьбу с последствиями коронавируса до 1,2% ВВП. На день 9 президент сообщает, что правительство направило на минимизацию последствий до 1,2% ВВП. Минфин уточняет: прямой пакет мер — 2,8% ВВП, фискальная поддержка населения и отраслей — 6,5% ВВП. Минэкономразвития фиксирует объём антикризисной программы свыше 2 трлн руб.

На день 10 Кудрин оценивает дополнительную потребность бюджета в 4 трлн руб., в том числе 2–3 трлн руб. на прямые дотации бизнесу, и рекомендует Минфину увеличить заимствования. Минфин моделирует: при нефти Urals по $20/барр. Фонд национального благосостояния похудеет примерно на 2 трлн руб. (на начало месяца остаток — около 13 трлн руб.).

На день 11 президент анонсирует новую ипотечную программу с потребностью 6,5 млрд руб. в текущем году. Кабмин поручает первому вице-премьеру представить предложения по оптимизации расходов федерального бюджета. Из резервного фонда правительства выделено более 273 млн руб. на незарегистрированный препарат для детей с тяжёлой патологией. На день 12 президент заявляет, что правительство направило на минимизацию последствий до 1,2% ВВП (повторно, уже в другом контексте). Минфин Силуанов заявляет, что меры борьбы с пандемией обойдутся бюджету в 2,8% ВВП, а фискальная поддержка — в 6,5% ВВП.

**Затронутые темы:** бюджетные расходы, госдолг, ФНБ, стресс-тесты, региональные финансы, поддержка бизнеса, оптимизация бюджета.
**Причинно-следственные связи:** падение нефтяных доходов (узел 1) → уменьшение бюджетных поступлений → необходимость наращивания долга и расходования ФНБ. Рост безработицы (узел 4) → увеличение социальных выплат.

## Узел 6. Международный контекст и координация (дни 4–10)

Глобальный экономический фон оказывает сильное давление на внутренние процессы. На день 4 Еврогруппа согласовывает «экономический ответ» на рецессию — до €500 млрд могут быть выделены незамедлительно. Bloomberg со ссылкой на банки Уолл-стрит сообщает, что мировая экономика потеряет за два года $5 трлн. Россия призывает снять многосторонние и односторонние санкции против развивающихся стран. Министр энергетики РФ Новак на заседании ОПЕК+ заявляет, что спрос на нефть упал столь серьёзно, что всем странам-производителям необходимо объединить усилия.

На день 8 Россия и ОПЕК+ заключают крупнейшую сделку об ограничении добычи нефти на

In [8]:
show("2021-10-20")

---
## calm_2021 — 2021-10-20

Вот единый нарративный таймлайн, собранный из предоставленных тематических осей и сгруппированный по смысловым событийным узлам.

---

### Узел 1: Глобальный сырьевой шок и укрепление рубля (День 1 — День 3)

Период открывается серией исторических рекордов на сырьевых рынках. Цена газа в Европе достигает абсолютного максимума — $1920 за 1 тыс. куб. м, после чего, на фоне заявлений о готовности стабилизировать рынок, резко падает до $1110–1250. Цена энергетического угля в Европе превышает $300 за тонну, побив рекорд 2008 года. Нефть Brent поднимается выше $84 за баррель (впервые с октября 2018 года), WTI — до $80,52. Рублёвая цена российской нефти Urals обновляет исторический максимум, достигая 5,87 тыс. руб. за баррель (рост более чем на 23% с конца предыдущего периода).

На этом фоне происходит резкое укрепление рубля. Курс евро падает до 83 руб. (минимум с середины предыдущего года), курс доллара — ниже 72 руб. Банк России выражает озабоченность ростом европейских цен на энергоносители, так как это усиливает импортируемую инфляцию. Сохраняется сигнал о возможности дальнейшего повышения ключевой ставки (текущий уровень — 6,75%), при этом регулятор отмечает, что двузначных значений не потребуется.

**Темы узла:** Глобальный энергетический кризис, укрепление рубля, цены на сырьё, рост импортируемой инфляции.
**Связи:** Взлёт нефтегазовых доходов приводит к укреплению рубля (налоговый период, продажа валюты экспортёрами). Рост цен на газ в Европе — ключевой триггер для всего периода.

---

### Узел 2: Инфляционное давление и реакция властей (День 1 — День 8)

Этот узел развивается параллельно и становится центральным для внутренней повестки. Опубликованы данные по инфляции за предыдущий месяц: годовая — 7,4%, месячная — 0,6%. Продукты дорожают на 9,2% год к году, плодоовощная продукция — на 15,2%. Минэкономразвития резко повышает годовой прогноз инфляции с 5,8% до 7,4%, основной причиной названо ускорение продовольственной инфляции (нетипичное для сентября подорожание овощей и мяса). Недельная инфляция ускоряется до 7,63%.

Корзина продовольствия демонстрирует взрывной рост: за неделю огурцы дорожают на 10,9%, помидоры — на 9,7%, картофель — на 4,4%, яйца — на 3,3%, курица — на 1,5%. С начала года накопленный рост картофеля составляет 38,9%, моркови — 31,4%, курицы — 22,6%.

Реакция властей многоуровневая. Президент признаёт, что цены на продовольствие выросли более чем на 7,5%, и связывает это с ситуацией на глобальных рынках энергоресурсов, называя это посткризисной волатильностью. Низкие доходы граждан обозначены как главная угроза стабильного развития; правительству поручено разработать дополнительные меры поддержки. Премьер-министр поручает вице-премьеру скоординировать работу по сдерживанию цен. Сеть «Пятерочка» после предупреждения ФАС снижает цены на морковь и капусту. Подкомиссия по таможенно-тарифному регулированию поддерживает продление льготы на ввоз сахара.

Банк России отмечает, что разовые проинфляционные факторы могут дольше удерживать инфляционные ожидания на повышенном уровне, и намекает на пересмотр прогноза по инфляции в сторону повышения.

**Темы узла:** Продовольственная инфляция, реакция правительства, административное регулирование цен, ухудшение прогнозов.
**Связи:** Ускорение инфляции — прямое следствие глобального сырьевого шока (удорожание удобрений, логистики, энергоносителей). Это вынуждает власти переходить от прогнозов к административным мерам.

---

### Узел 3: Рынок труда, доходы и социальные меры (День 7 — День 14)

На фоне высокой инфляции обостряются проблемы рынка труда и доходов населения. Уровень безработицы снижается до 4,4%, но ритейлеры и онлайн-компании сообщают об остром дефиците персонала (мерчендайзеры, кассиры, сборщики, курьеры). Причины: отток мигрантов, уход старшего поколения из-за пандемии, переход работников в курьерские службы. Число вакансий для курьеров и сборщиков выросло почти в 7 раз по сравнению с предыдущим месяцем. Доход водителя-курьера при полном графике достигает 200 тыс. руб. в месяц, пешего — до 100 тыс. руб., что провоцирует переток кадров из стационарной торговли в доставку.

В социальной сфере принимаются решения, привязанные к фактической инфляции. Президент предлагает индексировать материнский капитал с начала следующего года по фактической, а не прогнозной инфляции. В Москве увеличен прожиточный минимум (до 18 714 руб.) и минимальная пенсия (до 21 193 руб.). Утверждён рост пособия по уходу за первым ребёнком на 5,8% (до 7 493 руб.). Правительство направляет 28,3 млрд руб. на выплаты семьям с низкими доходами (дети от 3 до 7 лет) и 5,9 млрд руб. — многодетным.

Глава Дагестана сообщает о средней зарплате в регионе около 32 тыс. руб. (на 60% ниже среднероссийской) и росте числа живущих за чертой бедности на ~18% за последние 5 лет.

**Темы узла:** Дефицит кадров, рост зарплат в сегменте доставки, падение реальных доходов, увеличение социальных выплат.
**Связи:** Высокая инфляция давит на доходы, вынуждая правительство увеличивать социальные трансферты. Одновременно рост цен на сырьё укрепляет рубль, но не решает структурных проблем рынка труда.

---

### Узел 4: Денежно-кредитная политика и бюджетные доходы (День 8 — День 14)

Несмотря на укрепление рубля, инфляция остаётся выше цели, подготавливая почву для дальнейшего ужесточения ДКП. Аналитики ожидают повышения ключевой ставки на 0,5 п.п. (до 7,25%) на предстоящем заседании ЦБ. При этом регулятор фиксирует, что ставки по розничным кредитам практически не растут — банки жертвуют маржой для наращивания портфелей (средняя ПСК выросла незначительно, с 13,5% до 13,6%). Банк России объявляет о намерении обновить методику расчёта долговой нагрузки заёмщика, чтобы сделать выдачу беззалоговых кредитов менее выгодной.

На бюджетном фронте картина двойственная. Профицит федерального бюджета за три квартала превышает 1,4 трлн руб. Нефтегазовые доходы за период оцениваются примерно в 9 трлн руб. (~$125 млрд), их рост может составить $50 млрд к предыдущему периоду. Однако на фоне высокой заболеваемости коронавирусом ежедневные расходы государства на борьбу с пандемией составляют около 3,6 млрд руб., и Минфин готовит предложение о дополнительном финансировании более чем на 50 млрд руб.

Минэкономразвития вносит в правительство новые правила соглашений о защите капиталовложений: инвесторы смогут пересматривать условия при снижении курса рубля более чем на 20% или повышении ключевой ставки более чем на 3 п.п.

**Темы узла:** Ожидание повышения ключевой ставки, рост нефтегазовых доходов, профицит бюджета, расходы на борьбу с COVID-19, кредитная политика.
**Связи:** Сверхдоходы от нефти позволяют бюджету оставаться профицитным, но одновременно создают «голландскую болезнь» и укрепляют рубль. Высокая инфляция требует ужесточения ДКП, что может охладить экономику и ударить по потребительскому кредитованию.

---

### Узел 5: Вербальные интервенции и символические сигналы (День 8 — День 14)

В конце периода нарастает поток символических заявлений. Председатель ЦБ заявляет, что хранит сбережения в рублях, называя это выгодным и этичным. Курс доллара обновляет годовой минимум, опускаясь до 70,84 руб. Аналитики Citi выявляют устойчивое ослабление связи между ценами на нефть и курсом рубля после введения бюджетного правила: рост нефти на 10% теперь ведёт к укреплению рубля лишь на 1,6%.

Президент заявляет, что доллар теряет позиции резервной валюты, но Россия не заинтересована в полном отказе от расчётов в нём. Министр финансов предупреждает о рисках стагфляционного сценария в мировой экономике. Стоимость дизеля в ФРГ достигает рекордной отметки (€1,555 за литр). Розничные цены на овсяные хлопья в России достигают исторического максимума — 80,9 руб. за кг.

**Темы узла:** Номинальное укрепление рубля, вербальные сигналы регулятора, стагфляционные риски, рекорды розничных цен.
**Связи:** Укрепление рубля оказывается оторванным от реальной инфляции и воспринимается скорее как результат налогового периода и высоких цен на сырьё, а не как устойчивый тренд. Вербальные интервенции призваны стабилизировать ожидания.

---

### Общий контекст периода

Это был период **острого сырьевого суперцикла**, когда исторические рекорды цен на газ, уголь, нефть и металлы создали мощный проинфляционный фон как для мировой, так и для российской экономики. Внутри России высокая инфляция, особенно продовольственная, стала доминирующей социально-экономической проблемой, спровоцировав административное вмешательство в ценообразование и форсированное принятие мер социальной поддержки. Парадоксальным образом рост нефтегазовых доходов привёл к чрезмерному укреплению рубля, что в сочетании с дефицитом кадров на рынке труда и стагнацией реальных доходов сформировало неоднозначную конфигурацию: бюджетный профицит сосуществовал с инфляционным шоком для населения и подготовкой к дальнейшему ужесточению денежно-кредитной политики.

In [9]:
show("2022-03-25")

---
## shock_war — 2022-03-25

Вот единый нарративный таймлайн, собранный из 9 тематических осей. Все даты приведены к относительным (день 1, день 4 и т. д.), абсолютные упоминания месяцев и чисел заменены или удалены. Числовые показатели и имена собственные сохранены.

---

### Узел 1. Начало периода: экономическая война, ажиотажный спрос и первые прогнозы (дни 1–3)

В первые дни периода участники рынка и власти фиксируют шок от санкций. Представители России заявляют об объявлении экономической войны со стороны США и Запада. Министр финансов РФ сообщает о заморозке около **300 млрд долл.** золотовалютных резервов ЦБ. Канада вводит запрет на импорт российских нефтепродуктов. ЕС согласовывает четвёртый пакет санкций. Глава Всемирного банка полагает, что санкции окажут на мировую экономику большее влияние, чем сам конфликт.

На фоне валютной нестабильности ЦБ РФ временно запретил банкам взимать комиссию с физических лиц за выдачу наличной валюты. Курс доллара на Мосбирже в начале периода составлял **107,5 руб.**, евро — **117,8 руб.**, затем ЦБ понизил официальные курсы: доллар до **104,8 руб.**, евро до **115,93 руб.** Эксперты прогнозировали возможное падение доллара ниже **100 руб.** в течение недели. Цена газа на TTF в Европе снизилась до **~$1300 за тыс. куб. м** на фоне усиления ветра.

Опубликован макроэкономический опрос аналитиков Банка России: прогноз инфляции на текущий год повышен до **20%** (ранее 5,5%), ВВП снизится на **8%**, средний курс доллара — **110 руб.**, номинальная зарплата вырастет на **9,5%**. Реальный эффективный курс рубля за предыдущий месяц снизился на **1,7%**.

В продовольственном секторе начался ажиотажный спрос: фокус внимания россиян сместился на сахар. В Крыму и Приморье торговые сети ввели ограничения на продажу социально значимых товаров (сахар, мука, масло, крупы, соль) — не более **2–5 кг** в одни руки. Глава Крыма предложил ввести государственное регулирование цен на основные продукты питания сроком на полгода. Правительство РФ выделило **2,5 млрд рублей** на поддержку отечественных производителей хлеба. Сенат США одобрил пакет финансирования на **$1,5 трлн**, включая **$13,6 млрд** на помощь Украине.

По данным статистического ведомства, доля населения с доходами ниже границы бедности составила **11%** (около **16,1 млн человек**). Президент объявил о решении в ближайшее время увеличить МРОТ, прожиточный минимум, зарплаты бюджетников, социальные выплаты и пенсии. Глава Минтруда заявил, что работники компаний в простое должны получать не менее **2/3** средней зарплаты. Компании получат субсидии на переобучение сотрудников (**60 тыс. руб.** на человека). На портале «Работа в России» доступно свыше **1,6 млн вакансий**.

---

### Узел 2. Денежно-кредитная политика и динамика нефтяных котировок (дни 4–5)

На четвёртый день периода ЦБ РФ расширил временной диапазон для расчёта официального курса доллара (сделки до 16:30) и упростил порядок установления курса евро. Глава ЦБ впервые за два года отказалась от традиционной пресс-конференции после заседания по ставке, ограничившись письменным заявлением по ДКП. В условиях структурного дефицита ликвидности ЦБ объявил о проведении аукциона недельного репо; первоначально планировалось снизить лимит аукционов «тонкой настройки» с **3 трлн до 1 трлн руб.**, но на следующий день лимит был сохранён на уровне **3 трлн руб.** из-за повышенного спроса банков, а лимит депозитных аукционов снижен до **1 трлн руб.**

Минфин объявил о повышении экспортной пошлины на нефть с начала следующего месяца до **$61,2 за тонну**. Нефть Brent впервые с начала месяца опустилась ниже **$100 за баррель**, затем снизилась более чем на 5%, до **$102**. За неделю нефтяные котировки упали на **20–30%** (Brent, Urals) из-за отказа части потребителей от российской нефти, ожиданий поставок из Ирана и Венесуэлы, ухудшения эпидситуации в Китае. Нефть WTI опустилась ниже **$100 за баррель**.

На валютном рынке доллар на закрытие торгов составил **104 руб.**, евро — **114,6 руб.**, бивалютная корзина — **108,8 руб.** В начале следующих торгов доллар снизился на 0,96% до **103 руб.**, евро вырос на 1,19% до **116 руб.**

Производители кондитерских изделий (Ferrero, Mondelez, Mars, «Объединённые кондитеры», KDV) уведомили ритейлеров о повышении отпускных цен на **10–67%** из-за скачка курсов валют и роста себестоимости (какао подорожало на 60%, сахарная пудра — на 82%). Генпрокуратура и региональные власти начали проверки, заявив о недопустимости спекулятивного завышения цен.

---

### Узел 3. Инфляционный всплеск, реакция монетарных властей и первые бюджетные меры (дни 6–7)

На шестой день Минэкономики сообщило, что годовая инфляция ускорилась до **12,54%** (с 10,42% неделей ранее). Росстат: за неделю цены выросли на **2,1%** (замедление с 2,2% на предыдущей неделе); сахар подорожал на **12,8%** за неделю, с начала периода — на **20,4%**. Президент поручил ФАС и правительству постоянно мониторить цены.

Федеральная резервная система США повысила ключевую ставку на **25 б.п.** до **0,25–0,5%** и ухудшила прогноз роста ВВП США на текущий год до **2,8%** (с 4%). Председатель ФРС отметил крайнюю неопределённость, прогнозируя высокую инфляцию в США до середины года.

Розничные цены на бензин и дизельное топливо в РФ снизились впервые с сентября: бензин в среднем **50,89 руб./л**, дизтопливо — **54,97 руб./л**. Биржевые цены на бензин за неделю выросли: Аи-95 на **16,42%** (до **50 523 руб./т**), Аи-92 на **13,25%** (до **45 731 руб./т**). Премьер-министр РФ анонсировал меры поддержки экономики: **40 млрд руб.** на поддержание занятости и оборотный капитал системообразующих предприятий, **14 млрд руб.** на льготные кредиты для МСП.

Банк Англии повысил базовую ставку на **25 б.п.** до **0,75%** (третье повышение подряд). ФАС выступила против привязки цен внутренних контрактов к валютным курсам и внешним ценовым индикаторам. В России обсуждается повышение ставки по льготной ипотеке до **10–12%**. Профицит федерального бюджета РФ за два первых месяца составил **412,6 млрд руб.** Правительство планирует направить **546 млрд руб.** на социальную поддержку.

---

### Узел 4. Сохранение ключевой ставки и структурная перестройка (день 8)

Восьмой день стал ключевым для денежно-кредитной политики. Совет директоров ЦБ РФ сохранил ключевую ставку на уровне **20%** годовых. Глава ЦБ заявила:
- рост ставок — временная антикризисная мера, при стабилизации ставки будут снижаться;
- годовая инфляция вернётся к **4%** в 2024 году, но в текущем и следующем году она превысит прежние прогнозы, раскручивания инфляционной спирали не допустят;
- экономика входит в фазу масштабной структурной перестройки с временным периодом повышенной инфляции;
- новый макроэкономический прогноз будет представлен в следующем месяце;
- ЦБ и правительство работают над смягчением последствий для компаний с кредитами с плавающей ставкой.

ЦБ готов постепенно возобновлять торги на фондовом рынке: с понедельника откроются торги ОФЗ, при необходимости регулятор будет закупать их для поддержания стабильности. Средняя максимальная ставка по вкладам в крупнейших банках достигла **20,51%** годовых.

Банк Японии сохранил ультрамягкую ДКП (ключевая ставка **–0,1%**, доходность 10-летних госбумаг около **0%**), иена ослабла к доллару.

Минфин РФ предложил перераспределить расходы бюджета на текущий год в объёме **485,9 млрд руб.** Правительство выделит **78,8 млрд руб.** на меры поддержки занятости, **125 тыс. человек** переобучат за счёт бюджета. В пресс-релизе ЦБ отмечено: предыдущее резкое повышение ставки поддержало финансовую стабильность и предотвратило неконтролируемый рост цен.

---

### Узел 5. Меры поддержки рынка труда, социальные индексации и рост инфляционных ожиданий (дни 9–11)

Минтруд объявил об индексации социальных пенсий на **8,6%** с начала следующего месяца (затронет **4 млн человек**). Правительство выделило почти **40 млрд руб.** на поддержку рынка труда в регионах, из них **25,5 млрд руб.** — на создание временных рабочих мест и организацию общественных работ для **400 тыс. человек**. Дополнительно **14 млрд руб.** направлено на льготное кредитование МСП.

Международное агентство Fitch опубликовало прогноз: падение ВВП РФ на **8%**, инфляция **18%** в текущем году, курс доллара **135 руб.** к концу года. ЦБ повысил официальные курсы: доллар **104,68 руб.**, евро **115,6 руб.** Данные ЦБ: за предыдущий месяц средства населения в банках сократились на **1,2 трлн руб. (–3,5%)**, корпоративные кредиты выросли на **2,4%**, ипотека — на **2,2%**, потребительские кредиты — на **1,1%**.

ЦБ установил эквайринговые комиссии на уровне **1%** для социально значимых товаров и услуг (с середины следующего месяца до конца лета). Росздравнадзор зафиксировал снижение ажиотажного спроса на лекарства на **10%**. По данным опроса, при выборе новой работы **67%** россиян назвали стабильность главным фактором, **63%** — уровень зарплаты.

Птицефабрики уведомили ритейлеров о повышении отпускных цен на яйца; розничные цены уже выросли на **25%**, ожидается дальнейший рост до **40%**. Генпрокуратура поручила провести проверки производителей яиц и бытовой химии.

---

### Узел 6. Продовольственный кризис, региональные проблемы и начало стабилизации спроса (дни 12–13)

Росстат: недельная инфляция замедлилась до **1,93%** (с 2,09% неделей ранее). Цены на бензин снизились на **0,06%** (до **50,86 руб./л**). Сахар подорожал на **13,8%** за неделю, рис — на **3,9%**, гречка — на **3,3%**, плодоовощная продукция — на **3,8%** (лук репчатый +13,7%, помидоры +8,2%, бананы +7,8%). Инфляционные ожидания населения выросли до **18,3%** (второй по величине показатель за историю, рост на **4,8 п.п.**).

С начала периода отечественные автомобили подорожали на **20,7%**, иномарки — на **23,7%**. Промышленное производство в предыдущем месяце выросло на **6,3%** г/г.

Власти Приморья предупредили о подорожании сахара с **79 до 90–92 руб./кг** из-за роста биржевых цен и логистики. Правительство ввело временный запрет на вывоз сахара. Глава Алтая заявил о завышенных ценах на крупы, молочную продукцию и соль, поручив провести проверку УФАС. Росздравнадзор сообщил о росте цен на ряд жизненно важных лекарств до **40%**.

Минпромторг зафиксировал снижение объёма продаж сахара в крупных сетях на **29%** и возвращение спроса на другие повседневные товары к стандартному уровню, что говорит о сокращении ажиотажа.

Доля предпринимателей, оценивающих финансовое положение своего МСБ как плохое, выросла с **22% до 40%**. По оценкам, из России уехали **50–70 тыс.** IT-специалистов, прогнозируется вторая волна (**70–100 тыс.**) из-за отсутствия доступа к современным технологиям. Каждый третий IT-специалист ищет способы работать вне России с релокацией. Почти **96 тыс.** работников находились в простое, около **14 тыс.** — в отпуске без сохранения зарплаты. Задолженность по зарплате составила **930,9 млн руб.**, около **660 тыс.** человек зарегистрированы как безработные.

---

### Узел 7. Газовый ультиматум, укрепление рубля и глобальные риски (дни 13–14)

Президент поручил ЦБ и правительству за неделю определить порядок покупки рублей на внутреннем рынке для иностранных покупателей российского газа. После этого заявления рубль резко укрепился: доллар упал на **5,81 руб.** до **97,74 руб.**, евро — на **6,49 руб.** до **108,01 руб.** К концу периода ЦБ понизил официальные курсы: доллар до **96,05 руб.**, евро до **105,47 руб.** Рубль продолжал укрепляться на фоне возобновления торгов российскими акциями.

Нефть Brent превысила **$120 за баррель** (рост на 2,9%). Цена газа в Европе превысила **$1400 за тыс. куб. м** (рост 25%), при этом объём газа в ПХГ значительно ниже прошлогоднего. Президент поддержал повышение ставки по льготной ипотеке с **7% до 12%** и увеличение лимита такой ипотеки. Премьер-министр заявил, что платежи по ипотеке, оформленной до повышения ключевой ставки, останутся на уровне конца предыдущего месяца.

Общий объём задолженности предприятий и физических лиц по кредитам с плавающей процентной ставкой оценивается примерно в **14 трлн руб.** Кабмин направит **7,5 млрд руб.** на программу возврата половины стоимости путёвки на детский отдых. Дополнительно выделено **2 млрд руб.** на увеличение поставок зерновых культур и удобрений в регионы.

Президенты США и Франции заявили о реальной перспективе глобального дефицита продовольствия из-за конфликта и санкций, указывая на угрозу посевной кампании. Участники рынка ожидают падения импорта алкоголя в Россию на **20–40%** в годовом выражении из-за девальвации рубля и логистических проблем. Потери зарубежных авиакомпаний от облёта территории РФ составляют минимум **$2–3 млрд в год**.

---

### Общий контекст периода

Рассмотренный 14-дневный период характеризовался глубоким санкционным шоком, резким ростом инфляции и инфляционных ожиданий, а также масштабным оттоком рублёвой ликвидности из банковской системы. Центральный банк РФ сохранил ключевую ставку на уровне 20%, сделав ставку на временное замораживание денежно-кредитных условий и постепенное возобновление торгов. Правительство и регионы реализовали серию фискальных мер поддержки занятости, социальных выплат и продовольственного рынка. К концу периода наметилось замедление недельной инфляции и снижение ажиотажного спроса, а заявление о переводе расчётов за газ в рубли вызвало резкое укрепление национальной валюты, однако общая неопределённость оставалась крайне высокой.

In [10]:
show("2023-05-01")

---
## calm_2023 — 2023-05-01

# Единый нарративный таймлайн по смысловым узлам

## Узел 1: Макроэкономическая ситуация и дилемма денежно-кредитной политики (дни 1–2)

В начале периода обозначился ряд фундаментальных противоречий. Глава ЦБ заявила о сохранении прогноза инфляции на текущий год на уровне 5–7% и об отсутствии планов по новым валютным ограничениям — возможны лишь точечные поправки. При этом регулятор продолжает работу по возврату замороженных международных резервов в долларах и евро; переговорная позиция сформирована ответными ограничениями на вывод средств нерезидентами недружественных стран.

Годовая инфляция на середину периода составила 3,15%, что породило дилемму: быстрое снижение годового показателя (по итогам предыдущего месяца – 3,5%) одновременно с усилением проинфляционных факторов. Аналитики ЦБ отметили, что сдерживающие факторы (высокий урожай зерновых, рост производства мяса и молока при экспортных ограничениях) могут быть временными. Рост цен производителей потребительских товаров постепенно усиливается.

На ипотечном рынке ЦБ вводит дополнительное резервирование и макропруденциальные надбавки по рискованным займам. Аналитики «Эксперт РА» оценили потери банков от регуляторного ужесточения в 80 млрд руб. прибыли.

В бюджетной сфере финансирование строительства и модернизации магистральной инфраструктуры в предыдущем квартале сократилось в 2,4 раза год к году (подрядчики получили 33,2 млрд руб.). Минтранс спрогнозировал повышение стоимости авиабилетов на внутренние рейсы на 15–30% в летнем сезоне из-за отсутствия решения о субсидировании (в предыдущем году выделено 100 млрд руб.); зафиксировано сокращение рейсов на 10 612. Профильный комитет Госдумы рекомендовал отклонить законопроект об увеличении МРОТ до 30 тыс. руб. из-за отсутствия средств в принятом бюджете (с начала года МРОТ проиндексирован на 6,3% – до 16 242 руб.).

Численность занятых в IT-отрасли за предыдущий год выросла на 12% (до 761 тыс. человек), зарплаты — на 19% (вдвое выше средних по другим отраслям). Количество вакансий с «гибридным» графиком выросло в 1,3 раза.

Внешний фон: страны G7 сохранили потолок цен на российскую нефть на уровне $60 за баррель; представитель МИД РФ назвал ценовые потолки фактором дестабилизации мировых энергорынков. Международных инвесторов беспокоит банковский кризис и возможная рецессия. Олег Дерипаска охарактеризовал снижение рейтинга казначейских бумаг США как «вершину айсберга» проблем с госдолгом.

## Узел 2: Инфляция, цены и потребительский рынок (дни 3–5)

В середине первой недели годовая инфляция продолжила замедление: на день 3 она опустилась до 2,82% (замедление из-за снижения темпа роста цен в сфере услуг и отсутствия роста цен на непродовольственные товары вторую неделю подряд). Глава ЦБ уточнила, что годовая инфляция в марте составила 3,5% и продолжит снижаться в апреле, но текущий месячный прирост цен с поправкой на сезонность находится вблизи 4%. Инфляционные ожидания населения и бизнеса остаются выше уровней 2018–2019 годов, когда инфляция была вблизи 4%. Для дальнейшего снижения ключевой ставки необходимо снижение проинфляционных рисков.

На валютном рынке рубль стабилен: курс доллара — 81,5375 руб./$1, юань — 11,85 руб. По итогам дня 4 на Московской бирже доллар — 81,56 руб. (–0,34 руб.), евро — 89,53 руб. (–0,05 руб.), нефть Brent — $82,20/барр. (–$2,40). В лидерах роста «голубых фишек»: Yandex (+5,12%), ЛУКОЙЛ (+4,68%), Татнефть (+2,9%), Полюс (+2,3%), Новатэк (+1,61%); в падении — Норникель (–0,68%).

Биржевые цены на бензин выросли до максимумов с начала года: Аи-95 — 53 878 руб./т (+7,4% за 10 дней), Аи-92 — 48 175 руб./т (+7,5%) на фоне сезона плановых ремонтов на НПЗ и повышенного спроса. Минэнерго и ФАС заявили об отсутствии предпосылок для роста розничных цен.

В продовольственном сегменте: жевательная резинка стала самым подорожавшим за год продуктом (+22,3% из-за удорожания упаковки и запрета ЕС на ввоз резиновой основы). Опубликованы доходы блогеров за первый квартал: блогеры-физлица зарабатывали в среднем 15,8 тыс. руб./мес. (+27%), самозанятые — 57,1 тыс. руб. (+35%), блогеры-ИП — 206,7 тыс. руб. (–24%). Медианный доход на члена семьи в предыдущем году составил 22 тыс. руб. (в семьях с детьми — 18,3 тыс. руб.). Принят закон об увеличении налоговых вычетов: на обучение ребёнка — до 110 тыс. руб., на собственное обучение, лечение и лекарства — до 150 тыс. руб.

Спикер Госдумы сравнил инфляцию в России (2,82%) с европейскими странами (Венгрия — 25,6%, Латвия — 17,2%, Чехия — 16,5%), назвав действия ЦБ эффективными.

Помощник президента отметил, что правительство и ЦБ ищут пути решения проблемы повышенных долгосрочных ставок: доходность ОФЗ около 11% при целевой инфляции 4% даёт реальную доходность ~7%, что названо «экстремально высоким» из-за потери ликвидности после ухода иностранных инвесторов. Задача на текущий год — удержать инфляцию в диапазоне 5–7%; отклонение вниз также нежелательно.

## Узел 3: Валютное регулирование, ответные меры и структурные изменения (дни 6–9)

В конце первой — начале второй недели произошли важные институциональные изменения. ЦБ решил смягчить лимит на балансовую валютную позицию банков до ~50% (ранее проект предусматривал 20%); новая редакция ожидается во втором полугодии. Сбербанк повысил максимальную ставку по вкладу в юанях до 3,35% годовых (+0,72 п.п.) для сумм от 1 млн юаней на три года.

Президент подписал указ об ответных мерах на изъятие российских активов за рубежом: вводится внешнее управление над активами недружественных стран в РФ (внешний управляющий — Росимущество). Временное управление немедленно введено в отношении долей немецкой Uniper SE в «Юнипро» и финской Fortum в «Фортуме». Цель — сформировать компенсационный фонд. Доллар опустился ниже 81 руб., евро — до 89,31 руб., юань — до 11,75 руб.

В бюджетной сфере: новые регионы (ДНР, ЛНР, Запорожская и Херсонская области) получат суммарно более 410 млрд руб. из федерального бюджета (89% их совокупных доходов): ДНР — 171,1 млрд, ЛНР — 113,3 млрд, Запорожская — 65,2 млрд, Херсонская — 61 млрд, а также Крым (162 млрд) и Дагестан (133 млрд). Премьер-министр объявил о повышении МРОТ на 18,5% с будущего года, что затронет 4,8 млн граждан. Помощник президента спрогнозировал рост ВВП за текущий год на 1–2% и дефицит бюджета около 2% ВВП. «АвтоВАЗ» повышает цены на Lada в среднем на 2%.

На рынке труда зафиксирован дефицит IT-кадров: спрос вырос на 63% по сравнению с началом года, срок поиска кандидатов увеличился. Компании рассматривают аутсорсинг из Индии, Израиля и Китая; в перспективе дефицит может быть сглажен применением нейросетей. В Госдуму внесён законопроект о повышении НДФЛ до 30% для сотрудников, работающих из-за границы. По данным Superjob, каждый пятый россиянин (21,8%) хотел бы поменять профессию, но не знает, на какую.

Внешний контекст: в Кремле усомнились в продлении зерновой сделки; помощник министра финансов США предупредила о риске вторичных санкций для казахстанских компаний. Министр финансов РФ заявил, что кризис в США повлияет на Россию через снижение мирового спроса на экспорт.

## Узел 4: Предзаседание ЦБ и нарастание инфляционных сигналов (дни 10–11)

Перед заседанием ЦБ инфляционная картина стала более напряжённой. Инфляция с 18 по 24 апреля составила 0,10% (после 0,04% неделей ранее), с начала месяца — 0,31%, с начала года — 1,99%. Годовая инфляция замедлилась примерно до 2,49–2,55%. ВТБ с начала месяца прекратил выдачу ипотеки по сверхнизким ставкам; минимальная ставка по субсидированным программам от застройщиков теперь 4,5–8,5%. Ранее Сбербанк, Росбанк и другие банки также свернули или повысили ставки. ЦБ обязал банки с конца мая формировать дополнительные резервы по рискованной ипотеке, выданной после середины марта. Объём околонулевой ипотеки — около 800 млрд руб. (5% ипотечного портфеля).

Биржевые цены на бензин выросли ещё на 1,9% (Аи-92 до 49,7 тыс. руб./т) и 1,7% (Аи-95 до 54,9 тыс. руб./т) на фоне сообщений о возможном сокращении демпферных выплат нефтяникам (позже опровергнутых Минфином, но подтверждённых вице-премьером). Цена нефти Brent опустилась ниже $78 за баррель. Цены на уран превысили $52 за фунт из-за рисков сокращения предложения.

Топливно-энергетический комплекс попросил правительство пересмотреть сроки перехода на российское ПО и программно-аппаратные комплексы, предупреждая о миллиардных потерях и риске энергодефицита.

Законопроект о разовом взносе бизнеса (windfall tax) направлен на согласование: базовая ставка — 10% от превышения средней прибыли за 2021–2022 годы над показателем 2018–2019 годов; возможна досрочная уплата по ставке 5%. Сбербанк сообщил о намерении выплатить около 10 млрд руб. в рамках этого сбора. ВТБ освобождён от уплаты из-за рекордного убытка предыдущего года (756,8 млрд руб. по РСБУ).

Аналитики единодушно ожидали сохранения ключевой ставки на уровне 7,5% и «жёсткого» сигнала; зампред ЦБ подчеркнул, что решение о повышении не предрешено.

## Узел 5: Решение ЦБ по ключевой ставке и развёрнутый сигнал (день 12)

Центральный день периода — заседание совета директоров Банка России. ЦБ пятый раз подряд сохранил ключевую ставку на уровне 7,5% годовых. На заседании рассматривался и вариант повышения, но выбрано сохранение. Прогноз по инфляции на текущий год снижен до 4,5–6,5% (ранее 5–7%). Годовая инфляция на 24 апреля составила 2,5%. Сигнал о будущей ДКП ужесточён: «в условиях постепенного увеличения текущего инфляционного давления Банк России на ближайших заседаниях будет оценивать целесообразность повышения ставки для стабилизации инфляции вблизи 4% в следующем году».

Заявления главы ЦБ:
- Повышение ставки может потребоваться на ближайших заседаниях, если ускорение инфляции будет угрожать цели 4% в следующем году.
- Инфляция закрепится на 4% только в следующем году; двузначной инфляции не ожидается.
- Резкое снижение годовой инфляции не является основанием для снижения ставки.
- Прогнозируется постепенное нарастание инфляционного давления до конца года.
- ЦБ поддержал расширение адресных льготных ипотечных программ на вторичный рынок.
- В случае роста дефицита бюджета может потребоваться более жёсткая ДКП.
- ЦБ не видит тенденции к скорой разморозке заблокированных активов.
- Речи об ослаблении или ужесточении валютных ограничений сейчас не идёт.
- ЦБ рассматривает возможность раскрытия данных о ликвидности банков в разбивке на дружественные и недружественные валюты.

Баланс рисков по-прежнему смещён в сторону проинфляционных: геополитическая напряжённость, усиление внешних ограничений, усложнение логистики и расчётов. Обновлённый прогноз средней ключевой ставки: 7,3–8,2% на текущий год, 6,5–7,5% на следующий. Прогноз по цене на нефть Urals сохранён на уровне $55 за баррель на текущий и последующие два года.

Реакция рынка: курс рубля не изменился на решении; евро на Московской бирже опускался ниже 87 руб., доллар приблизился к 79 руб. Укрепление рубля связано с налоговым периодом и стабильной нефтяной конъюнктурой.

Одновременно опубликован «индекс шашлыка» (корзина с мясом, овощами, напитками): за год стоимость выросла на 2,62% (свинина), 3,66% (курица), 4,88% (баранина), 5,13% (говядина). При этом стоимость приготовления шашлыка (только мясо, соль, лук, перец) изменилась разнонаправленно: свинина –3,9%, курица –1,98%, баранина +5,56%, говядина +6,26%. Цена Brent поднялась до $79,02 за баррель.

В бюджетной сфере: правительство до 1 апреля следующего года приостановило публикацию статистики по нефти, газу и газовому конденсату. Глава ЦБ заявила, что российские эмитенты в текущем году должны выплатить по евробондам чуть более $7 млрд; внутренний рынок корпоративных облигаций (выросший на треть, до 13,3 трлн руб.) способен заместить это финансирование. Украина подала заявку на двухэтапное повышение тарифа на транзит российской нефти с €13,6 до €21 за тонну начиная с третьего квартала. В Минпромторге обсуждается запрет продажи картофеля, овощей и фруктов в полиэтиленовых сетках, что может привести к удорожанию примерно на 20%.

## Узел 6: Аналитика и пост-решение (дни 13–14)

На следующий после заседания день аналитики отметили, что ЦБ в пятый раз оставил ставку без изменений, несмотря на «ястребиный» сигнал; монетарная пауза продолжается. Оптовая цена свиной шеи достигла рекорда за пять лет — 450 руб./кг (+13% за неделю, +18% за год) из-за повышенного спроса в преддверии майских праздников. Замминистра энергетики Украины заявил о вынужденном увеличении тарифов на электроэнергию и необходимости $3,4 млрд для восстановления энергосистемы в текущем году.

---

## Общий контекст периода

Период характеризовался расхождением между низкой годовой инфляцией (около 2,5% к концу периода) и нарастанием текущего инфляционного давления (месячный рост цен около 4%), что поставило ЦБ перед дилеммой: сохранение ставки при «ястребином» сигнале. Внешний фон оставался напряжённым из-за санкционных ограничений, в том числе ответных мер (указ о внешнем управлении активами), и неопределённости на нефтяном рынке. Бюджетная политика сместилась в сторону ускорения расходов и адресной поддержки при одновременном поиске новых фискальных источников (windfall tax). На потребительском рынке фиксировался рост цен на бензин и отдельные продовольственные товары при сдержанной динамике зарплат, за исключением IT-сектора, где сохранялся дефицит кадров.